Data set 1: https://archive.ics.uci.edu/ml/datasets/individual+household+electric+power+consumption

These data set, is a data seet from Sceaux in France.
The data set containt a little bit more than 2million of meseare every minute between December 2006 and November 2010
The data set is compose by

In [7]:
import numpy as np
from ucimlrepo import fetch_ucirepo
import pandas as pd

individual_household_electric_power_consumption = fetch_ucirepo(id=235)
X = individual_household_electric_power_consumption.data.features

X.head()
X.info()

X['datetime'] = pd.to_datetime(X['Date'] + ' ' + X['Time'], dayfirst=True) #dayfirst= True because the format of the date is DD/MM/YYYY not MM/DD/YYYY
X.set_index('datetime', inplace=True)
X.drop(['Date', 'Time'], axis=1, inplace=True)

/home/rom1/PycharmProjects/ML_for_Data_Analytics_Project/.venv/lib/python3.10/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   object 
 1   Time                   object 
 2   Global_active_power    object 
 3   Global_reactive_power  object 
 4   Voltage                object 
 5   Global_intensity       object 
 6   Sub_metering_1         object 
 7   Sub_metering_2         object 
 8   Sub_metering_3         float64
dtypes: float64(1), object(8)
memory usage: 142.5+ MB


The data set is compose by 9 column, 2 of them are for de date and time and other are of type float.
## Description of the Dataset Columns

### Global_active_power (target)
This variable represents the total active power consumed by the household, averaged over one minute.
It is measured in kilowatts (kW) and corresponds to the real electrical power used by appliances to perform useful work such as lighting, heating, and running household devices.

### Global_reactive_power
This variable represents the total reactive power consumed by the household, averaged over one minute.
It is measured in kilowatts (kW). Reactive power does not perform useful work directly but is necessary to maintain electric and magnetic fields in devices such as motors and transformers.

### Voltage
This variable represents the average voltage of the household electrical system, measured in volts (V) during one minute.
Voltage is the electrical potential difference that drives current through the household electrical circuits.

### Global_intensity
This variable represents the average current intensity, measured in amperes (A).
It indicates how much electrical current flows through the household electrical system at a given time.

### Sub_metering_1
This variable represents the energy consumption of sub-metering zone 1, measured in watt-hours (Wh).
It mainly corresponds to the electricity used in the kitchen, including appliances such as the dishwasher, oven, and microwave.

### Sub_metering_2
This variable represents the energy consumption of sub-metering zone 2, measured in watt-hours (Wh).
It mainly corresponds to the electricity used in the laundry room, including appliances such as the washing machine, tumble dryer, refrigerator, and lighting.

### Sub_metering_3
This variable represents the energy consumption of sub-metering zone 3, measured in watt-hours (Wh).
It mainly corresponds to the electricity used by the electric water heater and the air conditioner.

According to the dataset description on the website, there are approximately **1.25% missing values** in the data.
Let's verify this information by checking how many missing values are present in each column of the dataset.

In [8]:
X = X.replace("?", np.nan)

missing_values = X.isnull().sum()
missing_percentage = (X.isnull().sum() / len(X)) * 100

missing_table = pd.DataFrame({
    "Column": X.columns,
    "Missing Values": missing_values.values,
    "Missing Percentage (%)": missing_percentage.values
})

missing_table

,Column,Missing Values,Missing Percentage (%)
0,Global_active_power,25979,1.251844
1,Global_reactive_power,25979,1.251844
2,Voltage,25979,1.251844
3,Global_intensity,25979,1.251844
4,Sub_metering_1,25979,1.251844
5,Sub_metering_2,25979,1.251844
6,Sub_metering_3,25979,1.251844


We observe that approximately **1.25% of the values are missing** in the dataset.
The number of missing values is identical across all columns, which suggests that these are **not isolated missing entries**, but rather **entire rows where all measurements are missing**.

This likely indicates that, at certain timestamps, the data recording failed and no measurements were captured for any variable.

In [9]:
# Count rows where ALL columns are missing
complete_missing_rows = X.isnull().all(axis=1).sum()

# Count rows where AT LEAST one column is missing
partial_missing_rows = X.isnull().any(axis=1).sum()

print(f"Number of rows where all columns are missing: {complete_missing_rows}")
print(f"Number of rows where at least one column is missing: {partial_missing_rows}")

Number of rows where all columns are missing: 25979
Number of rows where at least one column is missing: 25979


We can see that there are **25 979 rows where all columns are missing**,
and this is exactly the same as the number of rows with at least one missing value.

This confirms that the missing values are **not isolated in individual columns**,
but instead correspond to **entire rows with no recorded measurements**.

We need to check whether the missing values occur **consecutively** or not.
If there is only a **single consecutive missing value**, it can be easily replaced using the **average of the previous and next values**.
However, if multiple missing values occur consecutively, handling them will be **more challenging**.

In [12]:
# Create a mask: True if the row has at least one NaN
mask = X.isna().any(axis=1)

# Identify consecutive sequences
groups = (mask != mask.shift()).cumsum()

# Count the number of consecutive NaNs in each sequence
nan_streaks = mask.groupby(groups).sum()

# Keep only sequences that contain at least one NaN
nan_streaks = nan_streaks[nan_streaks > 0]

# Count how many times each length appears
streak_counts = nan_streaks.value_counts().sort_index()

# Create a DataFrame (index will be displayed)
df_streaks = pd.DataFrame({
    "Consecutive NaN Length": streak_counts.index,
    "Number of Sequences": streak_counts.values
})

# Display the table
print(df_streaks)

    Consecutive NaN Length  Number of Sequences
0                        1                   38
1                        2                   14
2                        3                    2
3                        4                    1
4                        6                    1
5                       21                    1
6                       24                    1
7                       33                    1
8                       38                    1
9                       43                    1
10                      47                    1
11                      70                    1
12                      83                    1
13                     891                    1
14                    2027                    1
15                    3129                    1
16                    3305                    1
17                    3723                    1
18                    5237                    1
19                    7226              

Most of the missing values occur in **long consecutive sequences**, with only a few short sequences of 1–4 missing rows.
This confirms that the missing data is **not isolated**, but rather corresponds to **entire periods with no recorded measurements**.
Special care will be needed to handle these long gaps during preprocessing or modeling.